# Dataset Validation & EDA

Exploratory analysis of the synthetic CPG/Retail dataset from Stage 1 Step 2.

**Purpose:** confirm visually that the intended business relationships are present, and
characterise the data future models will consume. No ML models are built here — that
starts at Step 4.

**Prerequisite:**

```powershell
uv run ari generate-data --profile dev --seed 42
uv run ari validate-data --profile dev
```

Data is read through `LocalDataRepository` rather than by opening Parquet directly —
the same abstraction every model and agent will use, so anything demonstrated here is
reachable by them too.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app.services.container import Container  # noqa: E402

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (11, 4.5)
pd.set_option("display.width", 160)

repo = Container().data_repository
DATA_ROOT = ROOT / "data" / "local"

manifest = json.loads((DATA_ROOT / "manifest.json").read_text())
print(f"dataset {manifest['dataset_version']}  seed {manifest['seed']}  "
      f"config {manifest['config_hash']}")
print(f"{manifest['start_date']} -> {manifest['end_date']}   "
      f"{manifest['total_rows']:,} rows across {len(manifest['row_counts'])} tables")
pd.Series(manifest["row_counts"]).sort_values(ascending=False).to_frame("rows")

## 1. Load a working sample

The full daily fact is ~6.7M rows. Two recent years is plenty for EDA and keeps every
cell below responsive.

In [ ]:
import datetime as dt

products = repo.get_products()
stores = repo.get_stores()
calendar = repo.get_calendar()
relationships = repo.get_product_relationships()

sales = repo.execute_query(
    """
    SELECT date, product_id, store_id, channel, units, regular_price, selling_price,
           discount_percentage, revenue, cost, gross_profit, promotion_flag, stockout_flag
    FROM sales_daily
    WHERE date >= DATE '2024-01-01'
    """,
    max_rows=4_000_000,
)
sales["date"] = pd.to_datetime(sales["date"])
sales = sales.merge(products[["product_id", "category", "brand"]], on="product_id")
sales = sales.merge(stores[["store_id", "region", "store_type"]], on="store_id")

print(f"{len(sales):,} rows | {sales['date'].min():%Y-%m-%d} -> {sales['date'].max():%Y-%m-%d}")
print(f"{sales['product_id'].nunique()} products x {sales['store_id'].nunique()} stores")
sales.head()

## 2. Revenue and volume trend

Look for: a coherent trend, weekly rhythm, and visible festival spikes. Flat, featureless
series would mean the seasonal and trend terms are not reaching the output.

In [ ]:
daily = sales.groupby("date").agg(
    revenue=("revenue", "sum"), units=("units", "sum"), profit=("gross_profit", "sum")
).reset_index()
daily = daily.merge(
    calendar[["date", "festival_flag", "holiday_flag", "festival_name"]].assign(
        date=lambda d: pd.to_datetime(d["date"])
    ),
    on="date", how="left",
)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(daily["date"], daily["revenue"], lw=0.7, color="#2b6cb0")
axes[0].plot(daily["date"], daily["revenue"].rolling(28).mean(), lw=2, color="#1a365d",
             label="28-day mean")
fest = daily[daily["festival_flag"].fillna(False)]
axes[0].scatter(fest["date"], fest["revenue"], s=6, color="#dd6b20", zorder=3, label="festival")
axes[0].set_ylabel("Revenue"); axes[0].legend(); axes[0].set_title("Daily revenue")

margin = daily["profit"] / daily["revenue"]
axes[1].plot(daily["date"], margin, lw=0.7, color="#38a169")
axes[1].plot(daily["date"], margin.rolling(28).mean(), lw=2, color="#22543d")
axes[1].set_ylabel("Gross margin"); axes[1].set_title("Margin — dips mark promotional periods")
plt.tight_layout()

In [ ]:
dow = sales.merge(calendar[["date", "day_name", "day_of_week"]].assign(
    date=lambda d: pd.to_datetime(d["date"])), on="date")
order = dow.sort_values("day_of_week")["day_name"].unique()
by_dow = dow.groupby("day_name")["units"].mean().reindex(order)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_dow.plot(kind="bar", ax=axes[0], color="#2b6cb0")
axes[0].set_title("Weekly rhythm — weekend lift"); axes[0].set_ylabel("Mean units/row")

monthly = sales.groupby([sales["date"].dt.month, "category"])["units"].mean().unstack()
monthly_idx = monthly.div(monthly.mean(axis=0), axis=1)
sns.heatmap(monthly_idx.T, cmap="RdYlGn", center=1.0, ax=axes[1], cbar_kws={"label": "index"})
axes[1].set_title("Seasonality by category (1.0 = category mean)")
axes[1].set_xlabel("Month")
plt.tight_layout()

## 3. Price → demand

The headline relationship. Compared **within product** — pooling across products would
mostly show that expensive categories sell fewer units, which is not elasticity.

In [ ]:
clean = sales[(sales["units"] > 0) & (~sales["stockout_flag"]) & (~sales["promotion_flag"])].copy()
clean["log_units"] = np.log(clean["units"])
clean["log_price"] = np.log(clean["selling_price"])

# Within-product deviations.
for col in ("log_units", "log_price"):
    clean[f"{col}_dev"] = clean[col] - clean.groupby("product_id")[col].transform("mean")

sample = clean.sample(min(40_000, len(clean)), random_state=42)
slope = np.polyfit(sample["log_price_dev"], sample["log_units_dev"], 1)[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hexbin(sample["log_price_dev"], sample["log_units_dev"], gridsize=45, cmap="Blues", mincnt=1)
xs = np.linspace(sample["log_price_dev"].min(), sample["log_price_dev"].max(), 50)
axes[0].plot(xs, slope * xs, color="#c53030", lw=2, label=f"slope = {slope:.2f}")
axes[0].set_xlabel("log price (within product)"); axes[0].set_ylabel("log units (within product)")
axes[0].set_title("Price vs demand — negative slope expected"); axes[0].legend()

elast = clean.groupby("category").apply(
    lambda g: np.polyfit(g["log_price_dev"], g["log_units_dev"], 1)[0]
    if len(g) > 500 else np.nan, include_groups=False
).dropna().sort_values()
elast.plot(kind="barh", ax=axes[1], color="#c53030")
axes[1].axvline(-1, ls="--", color="#4a5568", label="unit elastic")
axes[1].set_title("Pooled price sensitivity by category"); axes[1].legend()
plt.tight_layout()
print("Staples (Dairy, Packaged Food) should sit closer to zero than discretionary categories.")

### 3a. Recovered vs true elasticity

Ground truth is **not** reachable through the repository — it is loaded directly here,
which only validation and notebooks may do. No model may see this.

In [ ]:
truth = json.loads((DATA_ROOT / "ground_truth" / "elasticity.json").read_text())["values"]

clean["month"] = clean["date"].dt.to_period("M").astype(str)

def within(frame, col, *groups):
    out = frame[col].to_numpy(dtype=float)
    for g in groups:
        out = out - pd.Series(out).groupby(frame[g].to_numpy()).transform("mean").to_numpy()
    return out

rows = []
for pid, g in clean.groupby("product_id"):
    if len(g) < 800 or pid not in truth:
        continue
    y = within(g, "log_units", "store_id", "month")
    x = within(g, "log_price", "store_id", "month")
    if float(np.sum(x * x)) < 1e-9:
        continue
    rows.append({"product_id": pid, "true": truth[pid],
                 "recovered": float(np.sum(x * y) / np.sum(x * x)), "n": len(g)})

rec = pd.DataFrame(rows)
rec["abs_error"] = (rec["recovered"] - rec["true"]).abs() / rec["true"].abs()

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(rec["true"], rec["recovered"], alpha=0.55, s=28, color="#2b6cb0")
lims = [rec[["true", "recovered"]].min().min() - 0.2, rec[["true", "recovered"]].max().max() + 0.2]
ax.plot(lims, lims, ls="--", color="#c53030", label="perfect recovery")
ax.set_xlabel("True elasticity (hidden)"); ax.set_ylabel("Recovered (panel FE)")
ax.set_title("Elasticity recovery"); ax.legend()
plt.tight_layout()

print(f"products tested     : {len(rec)}")
print(f"median rel. error   : {rec['abs_error'].median():.1%}")
print(f"rank correlation    : {rec['true'].corr(rec['recovered'], method='spearman'):.3f}")
rec.sort_values("n", ascending=False).head(10)

## 4. Promotions

Two things to see: uplift rises with discount depth, and it **saturates**. If uplift were
linear in depth, Step 7's optimiser would simply choose the deepest discount every time.

In [ ]:
base_units = sales.loc[~sales["promotion_flag"], "units"].mean()
promo = sales[sales["promotion_flag"]].copy()
promo["band"] = pd.cut(promo["discount_percentage"], [0, 10, 20, 30, 100],
                       labels=["0-10%", "10-20%", "20-30%", "30%+"])
uplift = (promo.groupby("band", observed=True)["units"].mean() / base_units - 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
uplift.plot(kind="bar", ax=axes[0], color="#38a169")
axes[0].set_title("Uplift by discount depth"); axes[0].set_ylabel("Uplift vs baseline")

by_type = (promo.groupby("promotion_flag").size())  # placeholder guard
promos_tbl = repo.get_promotions()
promos_tbl.groupby("promotion_type")["discount_percentage"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="#805ad5")
axes[1].set_title("Mean discount by mechanic")

trade = repo.get_trade_promotions()
axes[2].hist(trade["roi"].clip(-1, 6), bins=40, color="#dd6b20", edgecolor="white")
axes[2].axvline(1.0, ls="--", color="#c53030", label="break-even")
axes[2].set_title("Trade promotion ROI"); axes[2].legend()
plt.tight_layout()

print(f"baseline mean units      : {base_units:.2f}")
print(uplift.round(3).to_string())
print(f"\ntrade promos below break-even: {(trade['roi'] < 1).mean():.1%} "
      f"— Step 7 needs bad promotions to allocate away from")

## 5. Stockouts: demand decline vs supply failure

The distinction the Root Cause agent must make in Step 17. Observed sales fall during a
stockout while *latent* demand does not — visible only because ground truth records both.

In [ ]:
scenarios = json.loads((DATA_ROOT / "ground_truth" / "scenario_config.json").read_text())["scenarios"]
stockouts = [s for s in scenarios if s["label"] == "stockout"]
sc = stockouts[0]

latent_parts = sorted((DATA_ROOT / "ground_truth" / "latent_demand").rglob("*.parquet"))
latent = pd.concat((pd.read_parquet(p) for p in latent_parts), ignore_index=True)
latent["date"] = pd.to_datetime(latent["date"])

sub = latent[(latent["product_id"] == sc["product_ids"][0])
             & (latent["store_id"].isin(sc["store_ids"]))]
ts = sub.groupby("date")[["latent_units", "observed_units"]].sum()

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(ts.index, ts["latent_units"].rolling(7).mean(), lw=2, color="#2b6cb0", label="true demand")
ax.plot(ts.index, ts["observed_units"].rolling(7).mean(), lw=2, color="#c53030", label="observed sales")
ax.axvspan(pd.Timestamp(sc["start_date"]), pd.Timestamp(sc["end_date"]),
           alpha=0.18, color="#dd6b20", label="injected stockout")
ax.set_title(f"Scenario D — {sc['product_ids'][0]}: supply failure, not demand collapse")
ax.set_ylabel("Units (7-day mean)"); ax.legend()
plt.tight_layout()

win = sub[(sub["date"] >= sc["start_date"]) & (sub["date"] <= sc["end_date"])]
print(f"during window — latent {win['latent_units'].sum():,}  "
      f"observed {win['observed_units'].sum():,}  "
      f"lost {win['lost_units'].sum():,} "
      f"({win['lost_units'].sum() / max(win['latent_units'].sum(), 1):.1%} suppressed)")

## 6. Cross-price relationships

Positive ⇒ substitutes, negative ⇒ complements. Estimated **holding the target's own
price constant** — same-category products share a cost index, so without that control
the target's own (large, negative) elasticity swamps the cross effect.

In [ ]:
cross_truth = json.loads((DATA_ROOT / "ground_truth" / "cross_elasticity.json").read_text())["values"]

pairs = sorted(((t, s, c) for t, srcs in cross_truth.items() for s, c in srcs.items()),
               key=lambda x: -abs(x[2]))[:10]

px = sales[["date", "product_id", "store_id", "selling_price", "units"]]
rows = []
for target, source, expected in pairs:
    t_rows = px[px["product_id"] == target]
    s_rows = px[px["product_id"] == source][["date", "store_id", "selling_price"]]
    m = t_rows.merge(s_rows, on=["date", "store_id"], suffixes=("_own", "_src"))
    m = m[(m["units"] > 0) & (m["selling_price_src"] > 0)]
    if len(m) < 300:
        continue
    m["month"] = m["date"].dt.to_period("M").astype(str)
    y = within(m.assign(v=np.log(m["units"])), "v", "store_id", "month")
    xs_ = within(m.assign(v=np.log(m["selling_price_src"])), "v", "store_id", "month")
    xo = within(m.assign(v=np.log(m["selling_price_own"])), "v", "store_id", "month")
    coef = np.linalg.lstsq(np.column_stack([xs_, xo]), y, rcond=None)[0]
    rows.append({"pair": f"{source} → {target}", "expected": expected,
                 "observed": float(coef[0]), "n": len(m)})

cp = pd.DataFrame(rows)
if not cp.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(cp["expected"], cp["observed"], s=60, color="#805ad5")
    lim = [min(cp[["expected", "observed"]].min()) - 0.2, max(cp[["expected", "observed"]].max()) + 0.2]
    ax.plot(lim, lim, ls="--", color="#c53030")
    ax.axhline(0, color="#a0aec0", lw=0.8); ax.axvline(0, color="#a0aec0", lw=0.8)
    ax.set_xlabel("True cross-elasticity"); ax.set_ylabel("Recovered")
    ax.set_title("Cross-price recovery")
    plt.tight_layout()
    agree = (np.sign(cp['expected']) == np.sign(cp['observed'])).mean()
    print(f"sign agreement: {agree:.0%}")
cp.round(3)

## 7. Regional, channel and product performance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sales.groupby("region")["revenue"].sum().sort_values().plot(
    kind="barh", ax=axes[0, 0], color="#2b6cb0")
axes[0, 0].set_title("Revenue by region")

sales.groupby("channel")["revenue"].sum().sort_values().plot(
    kind="barh", ax=axes[0, 1], color="#38a169")
axes[0, 1].set_title("Revenue by channel")

prod_rev = sales.groupby("product_id")["revenue"].sum().sort_values(ascending=False)
cum = prod_rev.cumsum() / prod_rev.sum()
axes[1, 0].plot(np.arange(1, len(cum) + 1) / len(cum) * 100, cum.to_numpy() * 100,
                lw=2, color="#805ad5")
axes[1, 0].axhline(80, ls="--", color="#c53030")
axes[1, 0].set_xlabel("% of products"); axes[1, 0].set_ylabel("% of revenue")
axes[1, 0].set_title("Revenue concentration — hero SKUs and a long tail")

sales.groupby("category")["gross_profit"].sum().div(
    sales.groupby("category")["revenue"].sum()).sort_values().plot(
    kind="barh", ax=axes[1, 1], color="#dd6b20")
axes[1, 1].set_title("Gross margin by category")
plt.tight_layout()

share_80 = float((cum <= 0.8).mean())
print(f"{share_80:.0%} of products generate 80% of revenue "
      f"— this skew is why WMAPE beats MAPE for forecasting")

## 8. Data quality — gold vs bronze

Gold is clean by construction. Bronze carries deliberately injected defects so the
quality framework has real problems to catch.

In [ ]:
injected = manifest.get("data_quality_injected", {})
flat = [{"table": t, "issue": i, "rows": n} for t, iss in injected.items() for i, n in iss.items()]
print(f"total injected defects (bronze only): {manifest.get('data_quality_total', 0):,}\n")

print("gold integrity:")
print(f"  null product_id      : {int(sales['product_id'].isna().sum())}")
print(f"  non-positive price   : {int((sales['selling_price'] <= 0).sum())}")
print(f"  negative units       : {int((sales['units'] < 0).sum())}")
rev_gap = (sales['revenue'] - sales['units'] * sales['selling_price']).abs().max()
print(f"  max revenue mismatch : {rev_gap:.4f}")
print(f"  stockout rate        : {sales['stockout_flag'].mean():.2%}")
print(f"  zero-unit day rate   : {(sales['units'] == 0).mean():.2%}")
pd.DataFrame(flat).sort_values("rows", ascending=False)

## 9. Validation summary

The authoritative pass/fail lives in `validation_results.json`, produced by
`uv run ari validate-data`. Reproduced here for convenience.

In [ ]:
results_path = DATA_ROOT / "validation_results.json"
if results_path.exists():
    res = json.loads(results_path.read_text())
    print(f"OVERALL: {'PASS' if res['passed'] else 'FAIL'}\n")
    rel = pd.DataFrame(res["relationships"])[
        ["name", "status", "observed", "expected", "tolerance", "sample_size"]]
    display(rel)
    chk = pd.DataFrame(res["checks"]["results"])
    print(f"\ninvariants: {res['checks']['summary']}")
    display(chk[chk["status"] != "PASS"] if (chk["status"] != "PASS").any()
            else chk.head(10))
else:
    print("Run: uv run ari validate-data --profile dev")

## Findings

1. **Trend, weekly rhythm and festival peaks are all present** — the forecasting model in
   Step 5 has real structure to learn rather than noise.
2. **Price and demand move in opposite directions within product**, and staples are
   visibly less elastic than discretionary categories.
3. **True elasticity is recoverable** with store and month fixed effects on
   non-promotional, in-stock rows. This is the property that makes Step 8 falsifiable.
4. **Promotional uplift rises with depth and saturates** — Step 7's optimiser faces a
   genuine allocation problem, not a corner solution.
5. **Trade promotion ROI straddles break-even**, so there is something to allocate away from.
6. **Stockouts suppress observed sales while latent demand holds** — the distinction the
   Root Cause agent must make.
7. **Cross-price signs match the declared relationships** once own price is controlled for.
8. **Revenue is heavily concentrated** in a minority of SKUs, which is why WMAPE is the
   honest forecasting metric.
9. **Gold is clean; bronze is not** — by design.

### Next

Step 3 (local database / repository), then Step 4 (baseline sales model), which is the
first model to be scored against the ground truth explored here.